# 04_05 – Разбор 147 пар повторного C40

Сравниваем все 294 исходные строки, включая списки времен и ВПП, метки, идентификаторы и справочные поля. Исходные строки извлекаются по позициям из рабочей копии Parquet; CSV используется для просмотра результатов. Решения об исправлении или удалении пока не принимаем.

In [1]:
# Подключаем разбор полных исходных записей в отдельном ядре
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents]
          if (p/'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
sys.path.insert(0,str(ROOT/'europe/scripts'))
import review_c40_pairs as analysis

# Проверяем контрольную сумму и сравниваем каждый исходный столбец внутри пары
pairs,differences=analysis.main()
display(pairs.groupby(['airport','label_pattern']).size().rename('pairs').reset_index())

{
  "input_sha256": "ddf11438bab44137242ffcf4da928418e116efd46defce58dc8a388b67942b6d",
  "input_unchanged": true,
  "pairs_reviewed": 147,
  "rows_reviewed": 294,
  "all_pairs_year": 2019,
  "all_pairs_same_callsign": true,
  "all_pairs_same_registration": true,
  "all_pairs_sequential_identifier_suffix": true,
  "all_pairs_same_c40_geometry": true,
  "label_patterns": {
    "both_false": 144,
    "early_true_late_false": 3
  },
  "pairs_with_different_typecodes": 0,
  "pairs_with_different_ILS_lists": 44,
  "all_late_records_one_attempt": true,
  "all_late_first_attempts_after_early_landing": true,
  "landing_gap_minutes": {
    "min": 12.266666666666667,
    "median": 17.483333333333334,
    "max": 26.316666666666666
  },
  "pairs_with_c40_after_early_first": 3,
  "rows_changed": 0,
  "rows_deleted": 0,
  "trajectories_available": false,
  "physical_identity_confirmed": false,
  "decision": "retain both pending independent validation"
}


,airport,label_pattern,pairs
0,EDDF,both_false,9
1,EDDF,early_true_late_false,1
2,EDDM,both_false,3
3,EDDM,early_true_late_false,1
4,EGCC,both_false,3
5,EGLL,both_false,27
6,EGSS,both_false,3
7,ESSA,both_false,3
8,LEMD,both_false,48
9,LFPG,both_false,10


In [2]:
# Показываем все 147 пар с временами и исходными метками
columns=['pair_id','airport','early_flight_id','late_flight_id','early_landing','late_landing',
         'landing_gap_minutes','early_n_attempts','late_n_attempts','early_GoA','late_GoA',
         'early_c40_to_first_minutes','late_c40_to_first_minutes','same_typecode','same_ILS']
with pd.option_context('display.max_rows',None,'display.max_columns',None):
    display(pairs[columns])

,pair_id,airport,early_flight_id,late_flight_id,early_landing,late_landing,landing_gap_minutes,early_n_attempts,late_n_attempts,early_GoA,late_GoA,early_c40_to_first_minutes,late_c40_to_first_minutes,same_typecode,same_ILS
0,1,EGLL,ETH710_2121,ETH710_2122,2019-01-09 19:34:24+00:00,2019-01-09 19:46:40+00:00,12.266667,1,1,False,False,12.700000,24.966667,True,True
1,2,LFPG,QTR7LN_4796,QTR7LN_4797,2019-01-30 05:07:35+00:00,2019-01-30 05:24:27+00:00,16.866667,1,1,False,False,16.133333,34.850000,True,True
2,3,EDDF,DAH2070_3432,DAH2070_3433,2019-02-01 11:33:24+00:00,2019-02-01 11:53:05+00:00,19.683333,1,1,False,False,15.983333,35.666667,True,True
3,4,LEMD,IBE3105_4368,IBE3105_4369,2019-07-02 20:56:18+00:00,2019-07-02 21:12:47+00:00,16.483333,1,1,False,False,13.100000,31.733333,True,False
4,5,LEMD,AEA6066_6712,AEA6066_6713,2019-06-18 15:36:02+00:00,2019-06-18 15:51:51+00:00,15.816667,1,1,False,False,17.866667,35.333333,True,False
5,6,LEMD,IBS37HF_10828,IBS37HF_10829,2019-07-02 20:52:51+00:00,2019-07-02 21:12:17+00:00,19.433333,1,1,False,False,-11.683333,10.233333,True,False
6,7,LEMD,IBE2001_11660,IBE2001_11661,2019-11-12 20:16:54+00:00,2019-11-12 20:33:58+00:00,17.066667,1,1,False,False,12.166667,30.983333,True,False
7,8,LEMD,IBE0553_21627,IBE0553_21628,2019-05-08 08:56:09+00:00,2019-05-08 09:13:28+00:00,17.316667,1,1,False,False,12.016667,31.366667,True,True
8,9,LEMD,AEA7708_28083,AEA7708_28084,2019-08-21 20:07:49+00:00,2019-08-21 20:24:17+00:00,16.466667,1,1,False,False,12.300000,34.750000,True,False
9,10,LEMD,IBE10HD_33600,IBE10HD_33601,2019-10-02 09:21:47+00:00,2019-10-02 09:38:50+00:00,17.050000,1,1,False,False,11.283333,30.166667,True,True


In [3]:
# Отдельно разбираем пары, затрагивающие положительную целевую метку
with pd.option_context('display.max_rows',None,'display.max_columns',None):
    display(pairs.loc[pairs.label_pattern!='both_false'])

# Различия справочных полей не приравниваем к смене физического самолета
display(differences.groupby('field').size().rename('pairs_with_difference').reset_index())
display(differences.loc[differences.field=='typecode'])

,pair_id,icao24,airport,C40_CROSS_TIME,early_position,late_position,early_flight_id,late_flight_id,early_callsign,late_callsign,early_registration,late_registration,early_typecode,late_typecode,early_landing,late_landing,early_first_attempt,early_last_attempt,late_first_attempt,late_last_attempt,early_n_attempts,late_n_attempts,early_GoA,late_GoA,label_pattern,landing_gap_minutes,early_landing_to_late_first_minutes,early_last_attempt_to_landing_minutes,late_last_attempt_to_landing_minutes,early_c40_to_first_minutes,late_c40_to_first_minutes,sequential_id_suffix,same_callsign,same_registration,same_typecode,same_ILS,same_c40_geometry,different_fields,decision
41,42,3c65c1,LFPG,2019-04-30 06:16:28+00:00,765326,765470,DLH77X_127043,DLH77X_127044,DLH77X,DLH77X,DAINA,DAINA,A20N,A20N,2019-04-30 07:02:28+00:00,2019-04-30 07:20:13+00:00,2019-04-30 06:35:43+00:00,2019-04-30 06:55:40+00:00,2019-04-30 07:20:13+00:00,2019-04-30 07:20:13+00:00,2,1,True,False,early_true_late_false,17.750000,17.750000,6.800000,0.000000,19.250000,63.750000,True,True,True,True,False,True,flight_id;landing_time;attempt_times;ILS;n_att...,retain_both_pending_independent_validation
45,46,3c6605,EDDM,2019-01-02 13:24:33+00:00,8417,8504,DLH2527_74481,DLH2527_74482,DLH2527,DLH2527,DAIPE,DAIPE,A320,A320,2019-01-02 14:09:47+00:00,2019-01-02 14:26:03+00:00,2019-01-02 13:45:33+00:00,2019-01-02 14:00:47+00:00,2019-01-02 14:25:55+00:00,2019-01-02 14:25:55+00:00,2,1,True,False,early_true_late_false,16.266667,16.133333,9.000000,0.133333,21.000000,61.366667,True,True,True,True,False,True,flight_id;landing_time;attempt_times;ILS;n_att...,retain_both_pending_independent_validation
130,131,506c2f,EDDF,2019-08-12 16:23:33+00:00,1528261,1528409,ADR118_209541,ADR118_209542,ADR118,ADR118,S5AAX,S5AAX,A319,A319,2019-08-12 16:59:40+00:00,2019-08-12 17:20:24+00:00,2019-08-12 16:40:08+00:00,2019-08-12 16:54:36+00:00,2019-08-12 17:18:44+00:00,2019-08-12 17:18:44+00:00,2,1,True,False,early_true_late_false,20.733333,19.066667,5.066667,1.666667,16.583333,55.183333,True,True,True,True,False,True,flight_id;landing_time;attempt_times;ILS;n_att...,retain_both_pending_independent_validation


,field,pairs_with_difference
0,GoA,3
1,ILS,44
2,__index_level_0__,147
3,attempt_times,147
4,c40_after_first_attempt,3
5,c40_after_landing,3
6,flight_id,147
7,landing_time,147
8,n_attempts,3


,pair_id,field,early_value,late_value


## Решение

Общие позывной, регистрация, геометрия C40 и соседние суффиксы идентификаторов поддерживают гипотезу связанных сегментов обработки. Они не доказывают истинный момент посадки или физическую идентичность полета. Удаление первой записи может потерять положительные метки; выбор второй записи автоматически не обоснован.

Для подтверждения нужны независимые траектории и правила выделения сегментов. [Репозиторий авторов](https://github.com/figuetbe/OSN23-GoA) прямо указывает, что использованные траектории не включены в него. В текущей базе их также нет.

Все записи сохранены; C40 и GoA не переписаны. [Подробный разбор](../../reports/analytical_sample/04_05_review.md), [поиск пар](04_04_c40_matching.ipynb).